# Blob pseudo-labels for synaptic puncta

Build binary `(1, 128, 128)` float32 pseudo-labels (1 = synapse,
0 = background) for 3-channel fluorescence patches loaded via
`PatchDataset`. Output is a downstream segmentation target.

- **Channels**: ch0 = pre-synaptic, ch1 = post-synaptic, ch2 = structural.
- **Pixel size**: 107 nm/px (60x confocal). Puncta 2-5 px; dendrites 4-18 px.
- **Guards**: `SWEEP=True` runs parameter sweeps; `GENERATE_ALL=True`
  produces labels for every patch.

## Pipeline

1. **Structural mask** -- dendrite detector + intensity threshold
   (somas) on the structural channel. Dendrite detector is selected
   by `cfg.dendrite_method`: `'meijering'` (Hessian ridge filter, for
   continuous filaments) or `'density'` (puncta-density pipeline, for
   structural channels where neurites appear as a chain of bright spots).
2. **LoG blob detection** on pre + post channels (Lindeberg 1998).
3. **z-score filter** -- SynQuant-inspired annular-background local-SNR test.
4. **Union** of surviving pre + post puncta (no co-localisation requirement;
   labels mark synaptic-marker puncta, not strictly co-localised synapses).
5. **Shape priors** + restriction to the dilated structural mask.

## Background

- **LoG** (`skimage.feature.blob_log`) returns `(row, col, sigma)` per
  detection. Puncta radius 1-2.5 px -> `min_sigma=0.7, max_sigma=1.8`
  (Lindeberg 1998).
- **Meijering ridge filter** (Meijering et al. 2004), designed for
  fluorescent neurite tracing. Dendrite radius 2-9 px ->
  `sigmas=range(2, 10)`.
- The structural mask and z-score each remove a different class of
  false positive (debris off-neurite, low-SNR candidates). Pre and
  post puncta are unioned (no co-localisation requirement).

## Patch-mode caveats

Patch-level LoG (small sigmas) is safe; ridge filters and global
thresholds are not. Dendrite + soma detection therefore run on the
**full reassembled image** and are sliced per patch.

| Issue | Fix |
|---|---|
| `blob_log` border attenuation within ~3*sigma_max | `cfg.log_exclude_border = 5` |
| Meijering border artefacts up to ~27 px on 128 px patches | run on full image (`compute_fullimage_structural_mask`) |
| Per-patch Otsu unstable when patches lack dendrites | one global threshold pooled across `N_CALIBRATION_IMAGES` images |
| Somas straddling patch edges fall below `soma_min_area` | detect on full image |
| Annular z-score (inner=3, outer=8 -> 17 px) fits inside 128 px | per-patch (no change) |

## Imports

In [ ]:
import os, sys, json
from datetime import datetime
from pathlib import Path

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

## Path setup

In [ ]:
NB_DIR = Path.cwd().resolve()
REPO_ROOT = NB_DIR
while REPO_ROOT.parent != REPO_ROOT and not (REPO_ROOT / '.git').is_dir():
    REPO_ROOT = REPO_ROOT.parent
ROOT = REPO_ROOT / 'src'
for p in (ROOT, REPO_ROOT):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))
print('REPO_ROOT:', REPO_ROOT)
print('ROOT     :', ROOT)

In [ ]:
from synaptic_ssl.pseudolabels.blobs import (
    BlobPseudoCfg,
    smooth_structural_channel,
    compute_global_meijering_threshold,
    compute_fullimage_structural_mask,
    meijering_response,
    density_response,
    compute_global_density_threshold,
    make_density_dendrite_mask,
    detect_blobs_log,
    score_blobs_zscore,
    generate_blob_pseudolabel,
    generate_pseudolabels_fullimage,
)
from synaptic_ssl.pseudolabels.viz import (
    show_3channel_grid,
    show_blob_overlay,
    show_scored_blobs,
    show_mask_overlay,
    show_pipeline_stages,
    plot_zscore_histogram,
)
from synaptic_ssl.training.sanity_batch import SANITY_TRAIN_INDICES
from synaptic_ssl.training.logging import setup_logger
from synaptic_ssl.utils_data.reassemble import reassemble_image, load_patch_records, ImageCache
from collections import defaultdict
from matplotlib.patches import Rectangle
from skimage.filters import threshold_otsu
from skimage.measure import label as cc_label, regionprops
from skimage.morphology import remove_small_objects


## Configuration

In [ ]:
# Data + IO
PATCH_ROOT  = REPO_ROOT / 'data' / 'patches_128'
OUTPUT_ROOT = REPO_ROOT / 'data' / 'pseudolabels_blob'
EXCLUDE_PATTERNS = ['KONTROLA']  # case-insensitive substrings to skip

# Sample sizes
N_CALIBRATION_IMAGES  = 8     # full images pooled for the Meijering threshold
N_VISUAL_REVIEW       = 24    # post-run review grid (multiple of 4)
SEED                  = 42

# ── Guard flags ──────────────────────────────────────────────
SWEEP        = False   # set True to run parameter sweeps
GENERATE_ALL = False   # set True to generate labels for ALL patches

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print('PATCH_ROOT :', PATCH_ROOT, ' (exists:', PATCH_ROOT.exists(), ')')
print('OUTPUT_ROOT:', OUTPUT_ROOT)
print(f'SWEEP={SWEEP}  GENERATE_ALL={GENERATE_ALL}')

In [ ]:
# All knobs in one place. Defaults are calibrated for 107 nm/px confocal,
# puncta diameter 2-5 px, dendrite width ~2-10 px.

cfg = BlobPseudoCfg(
    pre_channel=0, post_channel=1, structural_channel=2,

    # LoG on pre/post (puncta detection)
    log_min_sigma=0.7,
    log_max_sigma=1.8,
    log_num_sigma=5,
    log_threshold=0.005,    # very permissive; the z-score does the real filtering
    log_overlap=0.5,
    log_exclude_border=5,

    # Meijering for dendrites; covers thin (~2 px) to thick (~10 px) widths
    dendrite_sigmas=[1.0, 1.5, 2.0, 3.0, 5.0],
    dendrite_threshold=None,  # None = per-image Otsu on each image's Meijering response

    # soma (intensity threshold + size filter)
    soma_intensity_percentile=90.0,  # lowered from 99 to catch dimmer somas
    soma_min_area=2500,       # lowered from 5000 to catch smaller somas
                              # (real somas are 10-30 um diameter,
                              #  i.e. ~7000-60000 px^2; smaller values
                              #  may include bright dendrite junctions)
    soma_closing_radius=0,
    soma_fill_holes=False,

    # near-neuron zone width
    structural_dilation=4,    # ~ 0.43 um

    # shape priors
    min_size=3, max_size=40,
    min_fill=0.5, max_wh_ratio=4.0,

    # per-blob z-score on an annular background
    use_zscore=True,
    zscore_inner_radius=3,
    zscore_outer_radius=8,
    zscore_threshold=2.0,     # raw Gaussian z-score (mu_in - mu_bg) / sigma_bg

    # structural-channel smoothing (removes pixelation/hot pixels before Meijering)
    structural_smooth_method='median',    # 'none', 'gaussian', 'median', or 'tophat'
    structural_median_size=3,             # only used when method='median'
    structural_smooth_sigma=1.0,          # only used when method='gaussian'
    structural_tophat_radius=15,          # only used when method='tophat'; tuned in the sweep below

    # dendrite detector: 'meijering' = Hessian ridge filter, 'density' =
    # puncta-density pipeline (use for punctate structural channels).
    dendrite_method='meijering',
    # density-method knobs (only read when dendrite_method='density').
    density_input='tophat',                # 'raw' | 'tophat' | 'puncta'
    density_tophat_radius=15,
    density_sigma=6.0,                     # σ ~ puncta spacing
    density_global_threshold=None,         # None = per-image Otsu
    density_percentile=88.0,               # only used by compute_global_density_threshold
    density_min_cc_area=80,
    density_use_skeleton=True,
    density_skeleton_soma_min_area=2500,
    density_skeleton_soma_max_eccentricity=0.5,
    density_skeleton_prune=10,             # px
    density_skeleton_dilate=2,             # px; structural_dilation is applied on top
)
print(cfg)

## Output directory and logger

In [ ]:
RUN_TS   = datetime.now().strftime('%Y%m%d_%H%M%S')
save_dir = NB_DIR / 'outputs' / f'blob_pseudolabels_{RUN_TS}'
fig_dir  = save_dir / 'figures'
fig_dir.mkdir(parents=True, exist_ok=True)
print('save_dir =', save_dir)
print('fig_dir  =', fig_dir)

In [ ]:
logger = setup_logger('nb', save_dir / 'run.log')
logger.info(f'experiment = blob_pseudolabels')
logger.info(f'save_dir   = {save_dir}')
logger.info(f'patch_root = {PATCH_ROOT}  (exists={PATCH_ROOT.exists()})')
logger.info(f'output_root= {OUTPUT_ROOT}')

In [ ]:
# Persist the configuration alongside the figures.
(save_dir / 'config.json').write_text(
    json.dumps(
        {
            'cfg': cfg.__dict__,
            'patch_root': str(PATCH_ROOT),
            'output_root': str(OUTPUT_ROOT),
            'exclude_patterns': EXCLUDE_PATTERNS,
            'n_calibration_images': N_CALIBRATION_IMAGES,
            'n_visual_review': N_VISUAL_REVIEW,
            'sanity_train_indices': list(SANITY_TRAIN_INDICES),
            'seed': SEED,
        },
        indent=2, default=str,
    )
)
logger.info('config.json written')

## Plot saving helper

Every figure goes through `save_fig(fig, name)` which writes
`save_dir/figures/<name>.png` at 200 dpi.

In [ ]:
def save_fig(fig, name: str, dpi: int = 200, close: bool = False):
    """Save fig to fig_dir/<name>.png. Returns the resolved path."""
    out = fig_dir / f'{name}.png'
    fig.savefig(out, dpi=dpi, bbox_inches='tight')
    logger.info(f'[fig] {out.relative_to(save_dir)}')
    if close:
        plt.close(fig)
    return out

## Seed

In [ ]:
np.random.seed(SEED)
logger.info(f'seed = {SEED}')

## Data

Real patches loaded via `PatchDataset` -- items are `(C, H, W)`
float32 in `[0, 1]`. A synthetic fallback is defined below for the
case where `PATCH_ROOT` is missing (diagnostic only -- calibration
assumes real data).

In [ ]:
def make_synthetic_patch(rng, n_pre=20, n_post=20, n_dendrite=3, soma=True):
    """Fallback generator -- only used when the real patch dir is missing."""
    H = W = 128
    img = np.zeros((3, H, W), dtype=np.float32)
    img += rng.normal(0.05, 0.01, img.shape).clip(0, None)
    for _ in range(n_dendrite):
        y0 = rng.integers(20, 100); x0 = rng.integers(20, 100)
        dy, dx = rng.normal(0, 1), rng.normal(0, 1)
        n = np.hypot(dy, dx) + 1e-6; dy/=n; dx/=n
        for t in range(70):
            yy, xx = int(y0 + t*dy), int(x0 + t*dx)
            if 1 <= yy < H-1 and 1 <= xx < W-1:
                img[2, yy-1:yy+2, xx-1:xx+2] += 0.4
    if soma:
        cy, cx = 25, 25
        yy, xx = np.mgrid[:H, :W]
        img[2] += 0.9 * np.exp(-((yy-cy)**2 + (xx-cx)**2) / (2 * 9**2))
    for _ in range(n_pre):
        y, x = rng.integers(8, 120), rng.integers(8, 120)
        sy, ey = max(0, y-2), min(H, y+3); sx, ex = max(0, x-2), min(W, x+3)
        img[0, sy:ey, sx:ex] += 0.7
        oy = y + rng.integers(-1, 2); ox = x + rng.integers(-1, 2)
        sy, ey = max(0, oy-2), min(H, oy+3); sx, ex = max(0, ox-2), min(W, ox+3)
        img[1, sy:ey, sx:ex] += 0.7
    for _ in range(10):
        y, x = rng.integers(8, 120), rng.integers(8, 120)
        sy, ey = max(0, y-2), min(H, y+3); sx, ex = max(0, x-2), min(W, x+3)
        img[0, sy:ey, sx:ex] += 0.4
    return img.clip(0, 1)

In [ ]:
if PATCH_ROOT.exists():
    patch_records = load_patch_records(PATCH_ROOT, EXCLUDE_PATTERNS)
    logger.info(f'indexed {len(patch_records)} real patches from {PATCH_ROOT}')

In [ ]:
# Resolve the canonical sanity indices against the actual dataset size.
SANITY_INDICES = [min(int(i), len(patch_records) - 1) for i in SANITY_TRAIN_INDICES]
logger.info(f'SANITY_TRAIN_INDICES (resolved) = {SANITY_INDICES}')
for j, i in enumerate(SANITY_INDICES):
    name = patch_records[i]['filename']
    logger.info(f'  sanity[{j}] idx={i:5d}  {name}')

## Full-image structural mask cache

Dendrite + soma detection run on the full reassembled image
(~2304x2304) to avoid border artefacts and split somas, then are
sliced back into 128x128 tiles aligned with the patch grid. Each
image is reassembled lazily on first access.

In [ ]:
cache = ImageCache(PATCH_ROOT, patch_records)
image_to_patch_positions = cache.image_to_positions
available_image_indices = cache.available_image_indices
logger.info(
    f'{len(available_image_indices)} unique source images cover '
    f'{len(patch_records)} patches'
)

# Convenience aliases
get_full_image = cache.get_full_image
get_patch = cache.get_patch

_struct_full_cache: dict[tuple, dict] = {}


def _struct_cache_key(image_index: int) -> tuple:
    # repr(cfg) covers every dataclass field, so flipping
    # cfg.dendrite_method or any other knob invalidates automatically.
    return (int(image_index), repr(cfg))


def get_full_struct(image_index: int) -> dict:
    """Return the full-image structural dict for one source image.

    Returns a dict with the keys make_structural_mask emits for the
    active cfg.dendrite_method (always includes dendrite_response,
    dendrite_threshold, dendrite_mask, soma_mask, structural_mask,
    near_structural), plus full_image. Reassembles + computes on
    first access; cached afterwards. The dendrite threshold is per-image
    Otsu unless cfg.dendrite_threshold (meijering) or
    cfg.density_global_threshold (density) is set.

    Cache key includes repr(cfg), so any cfg edit invalidates -- avoids
    returning a stale Meijering mask after switching to density.
    """
    key = _struct_cache_key(image_index)
    if key in _struct_full_cache:
        return _struct_full_cache[key]
    full_image = get_full_image(image_index)
    struct = compute_fullimage_structural_mask(full_image, cfg)
    struct['full_image'] = full_image
    _struct_full_cache[key] = struct
    return struct

def get_patch_struct_slice(pos: int) -> dict:
    """Slice the cached full-image structural mask for one patch position."""
    rec = patch_records[pos]
    image_index = int(rec['image_index'])
    grid_row = int(rec['grid_row'])
    grid_col = int(rec['grid_col'])
    patch_size = int(rec['patch_size'])
    full = get_full_struct(image_index)
    y0 = grid_row * patch_size
    x0 = grid_col * patch_size
    sl = (slice(y0, y0 + patch_size), slice(x0, x0 + patch_size))
    # Slice every 2D array value that matches the full-image shape;
    # works for both meijering and density branches.
    full_shape = full['full_image'].shape[-2:]
    out = {}
    for k, v in full.items():
        if hasattr(v, 'shape') and v.shape[-2:] == full_shape and v.ndim == 2:
            out[k] = v[sl]
    return out

## Quick sanity checks

Visualisations on the canonical sanity batch
(`SANITY_TRAIN_INDICES`). Verify (1) channel order and (2) intensity
distribution before any parameter calibration.

In [ ]:
# Per-patch 3-channel grids on every sanity-batch index.
for j, i in enumerate(SANITY_INDICES):
    name = patch_records[i]['filename']
    p = get_patch(i)
    print(f'sanity[{j}]  idx={i}  {name}  range=[{p.min():.3f}, {p.max():.3f}]')
    fig, _ = show_3channel_grid(p)
    fig.suptitle(f'sanity[{j}]  idx={i}  {name}', y=1.02, fontsize=10)
    save_fig(fig, f'01_sanity_3channel_idx{i:05d}')
    plt.show()

In [ ]:
# Channel intensity histograms on the sanity batch only.
sub = [get_patch(i) for i in SANITY_INDICES]
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ch, name in enumerate(['pre (ch0)', 'post (ch1)', 'structural (ch2)']):
    vals = np.concatenate([p[ch].ravel() for p in sub])
    axes[ch].hist(vals, bins=80, log=True, color=['C2', 'C3', 'C0'][ch])
    axes[ch].set_title(f'{name}  (n={len(sub)} sanity patches)')
    axes[ch].set_xlabel('intensity'); axes[ch].set_ylabel('count (log)')
    axes[ch].grid(alpha=0.3)
fig.suptitle('Sanity batch -- channel intensity histograms', fontsize=11)
fig.tight_layout()
save_fig(fig, '02_sanity_channel_histograms')
plt.show()

## Structural-channel smoothing test

The structural channel often carries a periodic acquisition grid.
White top-hat (`image - opening(image, disk(r))`) extracts bright
features while removing the slow background -- a standard step for
neuronal processes in confocal fluorescence (Pathak et al. 2025,
§2.5.2: disk r=15 px on cortical neurons).

Single-config preview here; full radius sweep is in **Sweep: Dendrites**.

In [ ]:
# Close-up comparison on a single sanity patch (128x128).
demo_patch = get_patch(SANITY_INDICES[0])
ch_raw = demo_patch[cfg.structural_channel]
ch_processed = smooth_structural_channel(ch_raw, cfg)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(ch_raw, cmap='gray', vmin=0, vmax=0.5)
axes[0].set_title('raw structural (128x128)')
axes[1].imshow(ch_processed, cmap='gray', vmin=0, vmax=0.5)
axes[1].set_title(f'{cfg.structural_smooth_method} (r={cfg.structural_tophat_radius})')
axes[2].imshow(np.abs(ch_raw - ch_processed), cmap='hot')
axes[2].set_title('|raw - processed|  (removed background)')
for ax in axes.flat:
    ax.axis('off')
fig.suptitle(
    f'Structural smoothing test  (idx={SANITY_INDICES[0]}, '
    f'method={cfg.structural_smooth_method})',
    fontsize=11,
)
fig.tight_layout()
save_fig(fig, '03_structural_smoothing_test')
plt.show()

## Somas detection

Bright solid regions in the structural channel that Meijering
suppresses by design. Demo on the full image with the current
`soma_intensity_percentile` and `soma_min_area`. Sweep in
**Sweep: Somas** below.

In [ ]:
# Soma detection on the demo full image (standalone, no Meijering needed).
demo_idx = SANITY_INDICES[0]
demo_image_index = int(patch_records[demo_idx]['image_index'])
demo_full_img = get_full_image(demo_image_index)
demo_struct_channel = demo_full_img[cfg.structural_channel]

# Apply current soma config
soma_thr = float(np.percentile(demo_struct_channel, cfg.soma_intensity_percentile))
bright = demo_struct_channel > soma_thr
soma_mask = remove_small_objects(bright, min_size=cfg.soma_min_area)
lbl = cc_label(soma_mask)
n_cc = int(lbl.max())
max_area = int(max((p.area for p in regionprops(lbl)), default=0))

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(demo_struct_channel, cmap='gray', vmin=0, vmax=0.5)
axes[0].set_title('structural channel (full image)')
axes[1].imshow(soma_mask, cmap='gray')
axes[1].set_title(
    f'soma mask  (p={cfg.soma_intensity_percentile}, '
    f'min_area={cfg.soma_min_area})'
)
show_mask_overlay(
    demo_struct_channel, soma_mask,
    axes[2], color=(1.0, 0.2, 0.2), alpha=0.5, vmax=0.5,
)
axes[2].set_title(f'soma overlay  (#CC={n_cc}, max_area={max_area})')
for ax in axes.flat:
    ax.axis('off')
fig.suptitle(
    f'Somas detection  (image_index={demo_image_index}, '
    f'thr={soma_thr:.4f}, frac={soma_mask.mean():.2%})',
    fontsize=11,
)
fig.tight_layout()
save_fig(fig, '04_somas_detection')
plt.show()

print(f'soma_intensity_percentile = {cfg.soma_intensity_percentile}')
print(f'soma_min_area = {cfg.soma_min_area}')
print(f'threshold = {soma_thr:.4f}')
print(f'#CC = {n_cc},  max_area = {max_area},  mask_frac = {soma_mask.mean():.2%}')

## Dendrites detection

### Meijering threshold (per-image Otsu)

The pipeline uses per-image Otsu on the Meijering response of each
full reassembled image (Otsu, IEEE Trans SMC 1979). This cell is
diagnostic: it shows the pooled response distribution across
`N_CALIBRATION_IMAGES` images and the per-image Otsu threshold for
the demo image, which is what the demo + sweep cells below use.

In [ ]:
rng = np.random.default_rng(SEED)
calib_image_indices = rng.choice(
    available_image_indices,
    size=min(N_CALIBRATION_IMAGES, len(available_image_indices)),
    replace=False,
).tolist()
logger.info(f'calibration images: {calib_image_indices}')

# Reassemble each calibration image (cached) and pull its full structural channel.
calib_full_struct = [
    get_full_image(int(i))[cfg.structural_channel] for i in calib_image_indices
]

# Pooled distribution + per-image Otsu thresholds (diagnostic only;
# the pipeline uses per-image Otsu on each image at run time).
pooled = []
per_image_thresholds = []
for s in calib_full_struct:
    s_smooth = smooth_structural_channel(s, cfg)
    r = meijering_response(s_smooth, cfg.dendrite_sigmas)
    nz = r[r > 0]
    pooled.append(nz)
    per_image_thresholds.append(float(threshold_otsu(nz)) if nz.size else 0.0)
pooled = np.concatenate(pooled)
global_thr = float(threshold_otsu(pooled))
logger.info(f'per-image Otsu (mean) = {np.mean(per_image_thresholds):.4f} '
            f'(min={min(per_image_thresholds):.4f}, max={max(per_image_thresholds):.4f})')
logger.info(f'pooled-distribution Otsu (reference) = {global_thr:.4f}')

fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(pooled, bins=80, log=True, color='C0', alpha=0.7)
for thr in per_image_thresholds:
    ax.axvline(thr, color='C1', alpha=0.4, lw=1)
ax.axvline(global_thr, color='red', ls='--', label=f'pooled Otsu = {global_thr:.4f}')
ax.axvline(per_image_thresholds[0], color='C1', lw=1,
           label=f'per-image Otsu (n={len(per_image_thresholds)})')
ax.set_xlabel('Meijering response (>0 only)')
ax.set_ylabel('count (log)')
ax.set_title(f'Pooled Meijering response across {len(calib_full_struct)} full images')
ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout()
save_fig(fig, '05_meijering_calibration')
plt.show()

### Dendrite mask on the demo image

Per-image Otsu on the demo image's Meijering response, then the binary
dendrite mask. `demo_thr` is reused below in the threshold sweep.

In [ ]:
demo_struct_smooth = smooth_structural_channel(demo_struct_channel, cfg)
demo_response = meijering_response(demo_struct_smooth, cfg.dendrite_sigmas)
_demo_nz = demo_response[demo_response > 0]
demo_thr = float(threshold_otsu(_demo_nz)) if _demo_nz.size else 0.0
demo_dendrite_mask = demo_response > demo_thr

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(demo_struct_smooth, cmap='gray', vmin=0, vmax=0.5)
axes[0].set_title(f'structural (smoothed: {cfg.structural_smooth_method})')
axes[1].imshow(demo_response, cmap='hot')
axes[1].set_title(f'Meijering response (sigmas={list(cfg.dendrite_sigmas)})')
show_mask_overlay(
    demo_struct_channel, demo_dendrite_mask,
    axes[2], color=(0.2, 0.7, 1.0), alpha=0.45, vmax=0.5,
)
axes[2].set_title(
    f'dendrite overlay  (per-image Otsu={demo_thr:.3f}, '
    f'frac={demo_dendrite_mask.mean():.2%})'
)
for ax in axes.flat:
    ax.axis('off')
fig.suptitle(
    f'Dendrites detection  (image_index={demo_image_index})',
    fontsize=11,
)
fig.tight_layout()
save_fig(fig, '06_dendrites_detection')
plt.show()

## Density-method dendrite mask

For structural channels where the "dendrite" is a chain of dense bright
puncta. A punctum has `λ₁ ≈ λ₂ << 0` (blob), not `λ₁ ≈ 0, λ₂ << 0`
(ridge), so Hessian ridge filters match the wrong signal.

The pipeline: top-hat → Gaussian σ ≈ inter-spot spacing → threshold →
small-CC filter → optional skeletonise + graph-walk prune of short
leaf branches + dilate.

In [ ]:
# Density vs Meijering on the same demo image. Uses a throwaway cfg so
# the top-level cfg stays on dendrite_method='meijering'.
_density_cfg = BlobPseudoCfg(**{**cfg.__dict__, 'dendrite_method': 'density'})
_density = make_density_dendrite_mask(demo_struct_channel, _density_cfg)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Row 0: Meijering pipeline (recap)
axes[0, 0].imshow(demo_struct_smooth, cmap='gray', vmin=0, vmax=0.5)
axes[0, 0].set_title(f'structural (smoothed: {cfg.structural_smooth_method})')
axes[0, 1].imshow(demo_response, cmap='hot')
axes[0, 1].set_title(f'Meijering response (sigmas={list(cfg.dendrite_sigmas)})')
show_mask_overlay(
    demo_struct_channel, demo_dendrite_mask,
    axes[0, 2], color=(0.2, 0.7, 1.0), alpha=0.45, vmax=0.5,
)
axes[0, 2].set_title(
    f'Meijering mask  (Otsu={demo_thr:.3f}, frac={demo_dendrite_mask.mean():.2%})'
)

# Row 1: Density pipeline
axes[1, 0].imshow(_density['density_input_field'], cmap='gray')
axes[1, 0].set_title(
    f"density_input='{_density_cfg.density_input}' "
    f"(r={_density_cfg.density_tophat_radius})"
)
axes[1, 1].imshow(_density['density_response'], cmap='hot')
axes[1, 1].set_title(f'smoothed density (σ={_density_cfg.density_sigma})')
show_mask_overlay(
    demo_struct_channel, _density['dendrite_mask'],
    axes[1, 2], color=(1.0, 0.55, 0.0), alpha=0.45, vmax=0.5,
)
axes[1, 2].set_title(
    f"density mask  (thr={_density['dendrite_threshold']:.3f}, "
    f"frac={_density['dendrite_mask'].mean():.2%}, "
    f"skeleton={_density_cfg.density_use_skeleton})"
)
for ax in axes.flat:
    ax.axis('off')
fig.suptitle(
    f'Dendrite detector comparison  (image_index={demo_image_index})',
    fontsize=11,
)
fig.tight_layout()
save_fig(fig, '06b_dendrites_method_comparison')
plt.show()

print(
    f"Meijering: dendrite_mask frac = {demo_dendrite_mask.mean():.2%}\n"
    f"Density:   dendrite_mask frac = {_density['dendrite_mask'].mean():.2%}, "
    f"raw_density frac = {_density['raw_density_mask'].mean():.2%}"
)

## Blobs detection

### Step A -- combined structural mask

Now that both somas and dendrites are configured, compute the
full structural mask (dendrite OR soma, dilated) and show the
patch-level slice.

In [ ]:
# Full-image structural mask (combines dendrite + soma + dilation).
full_struct = get_full_struct(demo_image_index)

demo_rec = patch_records[demo_idx]
patch_size = int(demo_rec['patch_size'])
y0 = int(demo_rec['grid_row']) * patch_size
x0 = int(demo_rec['grid_col']) * patch_size

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
full_struct_channel = full_struct['full_image'][cfg.structural_channel]

axes[0, 0].imshow(full_struct_channel, cmap='gray', vmin=0, vmax=0.5)
axes[0, 0].set_title('structural channel (full)')
axes[0, 1].imshow(full_struct['dendrite_mask'], cmap='gray')
axes[0, 1].set_title(
    f'dendrite mask  (per-image Otsu={full_struct["dendrite_threshold"]:.3f}, '
    f'frac={full_struct["dendrite_mask"].mean():.2%})'
)
axes[0, 2].imshow(full_struct['soma_mask'], cmap='gray')
axes[0, 2].set_title(
    f'soma mask  (p={cfg.soma_intensity_percentile}, '
    f'min_area={cfg.soma_min_area}, '
    f'frac={full_struct["soma_mask"].mean():.2%})'
)
axes[1, 0].imshow(full_struct['dendrite_response'], cmap='hot')
axes[1, 0].set_title(f'{cfg.dendrite_method} response')
n_soma_cc = int(cc_label(full_struct['soma_mask']).max())
show_mask_overlay(
    full_struct_channel, full_struct['dendrite_mask'],
    axes[1, 1], color=(0.2, 0.7, 1.0), alpha=0.45, vmax=0.5,
)
axes[1, 1].set_title('dendrite overlay on structural')
show_mask_overlay(
    full_struct_channel, full_struct['soma_mask'],
    axes[1, 2], color=(1.0, 0.2, 0.2), alpha=0.5, vmax=0.5,
)
axes[1, 2].set_title(f'soma overlay on structural  (#CC={n_soma_cc})')
for ax in axes.flat:
    ax.add_patch(Rectangle((x0, y0), patch_size, patch_size,
                           edgecolor='lime', facecolor='none', linewidth=1.5))
    ax.axis('off')
fig.suptitle(
    f'Step A: full-image structural mask  '
    f'(image_index={demo_image_index}, demo patch outlined in lime)',
    fontsize=11,
)
fig.tight_layout()
save_fig(fig, '07_step_A_structural_mask')
plt.show()

In [ ]:
# Combined structural mask + dilation, sliced from the full-image computation.
demo = get_patch(demo_idx)
struct_dict = get_patch_struct_slice(demo_idx)
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(struct_dict['structural_mask'], cmap='gray')
axes[0].set_title(f'dendrite OR soma  (frac={struct_dict["structural_mask"].mean():.2%})')
axes[1].imshow(struct_dict['near_structural'], cmap='gray')
axes[1].set_title(
    f'after {cfg.structural_dilation}-px dilation  '
    f'(frac={struct_dict["near_structural"].mean():.2%})'
)
show_mask_overlay(demo.max(0), struct_dict['near_structural'],
                  axes[2], color=(0.2, 0.5, 1.0), alpha=0.4)
axes[2].set_title('overlay on composite')
fig.suptitle(
    f'Step A: structural mask + dilation, sliced from full image  '
    f'(idx={demo_idx})',
    fontsize=11,
)
fig.tight_layout()
save_fig(fig, '08_step_A_structural_dilation')
plt.show()

### Step B -- LoG blob detection on pre / post

In [ ]:
pre_blobs  = detect_blobs_log(demo[cfg.pre_channel],  cfg)
post_blobs = detect_blobs_log(demo[cfg.post_channel], cfg)
logger.info(f'pre  LoG candidates: {len(pre_blobs)}')
logger.info(f'post LoG candidates: {len(post_blobs)}')

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
show_blob_overlay(demo[cfg.pre_channel],  pre_blobs,  axes[0], color='lime')
axes[0].set_title(f'pre   ch{cfg.pre_channel}  ({len(pre_blobs)} candidates)')
show_blob_overlay(demo[cfg.post_channel], post_blobs, axes[1], color='magenta')
axes[1].set_title(f'post  ch{cfg.post_channel}  ({len(post_blobs)} candidates)')
fig.suptitle(f'Step B: LoG blob detection  (idx={demo_idx})', fontsize=11)
fig.tight_layout()
save_fig(fig, '09_step_B_log_blobs')
plt.show()

### Step C -- per-blob z-score (local-SNR test on an annular background)

For each LoG candidate compute the mean intensity inside the blob
disk and compare it against an *annular* background ring. The ratio
is converted to a z-score; only blobs with `z >= cfg.zscore_threshold`
survive.

In [ ]:
scored_pre  = score_blobs_zscore(demo[cfg.pre_channel],  pre_blobs,  cfg)
scored_post = score_blobs_zscore(demo[cfg.post_channel], post_blobs, cfg)

n_pre_kept  = sum(s['kept'] for s in scored_pre)
n_post_kept = sum(s['kept'] for s in scored_post)
logger.info(f'pre:  {n_pre_kept}/{len(scored_pre)} kept (z >= {cfg.zscore_threshold})')
logger.info(f'post: {n_post_kept}/{len(scored_post)} kept')

fig, ax = plt.subplots(figsize=(8, 4))
plot_zscore_histogram(scored_pre, scored_post, cfg.zscore_threshold, ax=ax)
fig.suptitle(f'Step C: z-score distribution  (idx={demo_idx})', fontsize=11)
fig.tight_layout()
save_fig(fig, '10_step_C_zscore_histogram')
plt.show()

In [ ]:
# Visual: kept (lime) vs rejected (red).
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
show_scored_blobs(demo[cfg.pre_channel],  scored_pre,  axes[0])
axes[0].set_title(f'pre  -- lime kept ({n_pre_kept}), red rejected')
show_scored_blobs(demo[cfg.post_channel], scored_post, axes[1])
axes[1].set_title(f'post -- lime kept ({n_post_kept}), red rejected')
fig.suptitle(f'Step C: z-score kept vs rejected  (idx={demo_idx})', fontsize=11)
fig.tight_layout()
save_fig(fig, '11_step_C_zscore_kept_vs_rejected')
plt.show()

### Step D -- full pipeline on every sanity patch

`generate_blob_pseudolabel` runs every step end-to-end. The 3x3 figure
shows every intermediate; the bottom-right is the final binary
pseudo-label. We render this figure for **every** sanity index so
all five canonical patches are saved for the thesis.

In [ ]:
for j, i in enumerate(SANITY_INDICES):
    name = patch_records[i]['filename']
    p = get_patch(i)
    struct_slice = get_patch_struct_slice(i)
    label_mask, intermediates, stats = generate_blob_pseudolabel(
        p, cfg, precomputed_struct=struct_slice,
    )
    print(f'sanity[{j}]  idx={i}  {name}')
    for k, v in stats.items():
        print(f'  {k:24s}: {v}')
    fig, _ = show_pipeline_stages(p, intermediates, label_mask)
    fig.suptitle(
        f'Step D: full pipeline  sanity[{j}]  idx={i}  {name}\n'
        f'label_px={stats["px_label"]}  frac={stats["frac_label"]:.3%}',
        fontsize=11,
    )
    save_fig(fig, f'12_step_D_pipeline_sanity{j}_idx{i:05d}')
    plt.show()

---
## Sweep: Somas

**Guarded by `SWEEP = True`.**

Sweep `soma_intensity_percentile` over the demo image to find the
value that highlights visible somas without picking up scattered
bright maxima inside dendrites.

In [ ]:
if SWEEP:
    soma_pcts = [85.0, 90.0, 95.0, 99.0]
    fig, axes = plt.subplots(1, len(soma_pcts), figsize=(4 * len(soma_pcts), 4))
    print(
        f"{'pct':>6s} | {'thr':>8s} | {'#CC kept':>10s} | "
        f"{'max area':>10s} | {'mask frac':>10s}"
    )
    for ax, pct in zip(axes, soma_pcts):
        thr = float(np.percentile(demo_struct_channel, pct))
        bright = demo_struct_channel > thr
        keep = remove_small_objects(bright, min_size=cfg.soma_min_area)
        lbl = cc_label(keep)
        n_cc = int(lbl.max())
        max_area = int(max((p.area for p in regionprops(lbl)), default=0))
        print(
            f'{pct:6.1f} | {thr:8.4f} | {n_cc:10d} | '
            f'{max_area:10d} | {keep.mean():10.2%}'
        )
        show_mask_overlay(demo_struct_channel, keep, ax,
                          color=(1.0, 0.2, 0.2), alpha=0.5, vmax=0.5)
        ax.set_title(
            f'p={pct}, thr={thr:.3f}\n#CC={n_cc}  max_area={max_area}',
            fontsize=9,
        )
        ax.axis('off')
    fig.suptitle(
        f'Soma percentile sweep  (image_index={demo_image_index}, '
        f'min_area={cfg.soma_min_area})',
        fontsize=11,
    )
    fig.tight_layout()
    save_fig(fig, '20_sweep_soma_percentile')
    plt.show()
else:
    print('SWEEP is False -- skipping soma sweep')

## Sweep: Dendrites

**Guarded by `SWEEP = True`.**

Two sweeps:
1. **Dendrite threshold** -- compare Otsu vs percentile thresholds.
2. **Top-hat radius** -- how smoothing radius affects the dendrite
   mask.

In [ ]:
if SWEEP:
    # ── Dendrite threshold sweep ──
    nz = demo_response[demo_response > 0]
    thr_candidates = {
        'otsu (per-image)': demo_thr,
        'p90': float(np.percentile(nz, 90)),
        'p95': float(np.percentile(nz, 95)),
        'p98': float(np.percentile(nz, 98)),
        'p99': float(np.percentile(nz, 99)),
    }

    print(f"{'method':>20s} | {'thr':>8s} | {'mask frac':>10s}")
    for name, thr in thr_candidates.items():
        frac = float((demo_response > thr).mean())
        print(f'{name:>20s} | {thr:8.4f} | {frac:10.2%}')

    fig, axes = plt.subplots(1, len(thr_candidates), figsize=(4 * len(thr_candidates), 4))
    for ax, (name, thr) in zip(axes, thr_candidates.items()):
        mask = demo_response > thr
        ax.imshow(mask, cmap='gray')
        ax.set_title(f'{name}\nthr={thr:.3f}  frac={mask.mean():.1%}', fontsize=9)
        ax.axis('off')
    fig.suptitle(
        f'Dendrite threshold sweep  (image_index={demo_image_index}, '
        f'sigmas={list(cfg.dendrite_sigmas)})',
        fontsize=11,
    )
    fig.tight_layout()
    save_fig(fig, '21_sweep_dendrite_threshold')
    plt.show()
else:
    print('SWEEP is False -- skipping dendrite threshold sweep')

In [ ]:
if SWEEP:
    # ── Top-hat radius sweep on the demo image's structural channel ──
    demo_img0 = get_full_image(demo_image_index)
    demo_struct_raw = demo_img0[cfg.structural_channel]

    tophat_radii = [0, 8, 12, 15, 20]
    n_cand = len(tophat_radii)

    fig, axes = plt.subplots(3, n_cand, figsize=(4 * n_cand, 12))
    for j, radius in enumerate(tophat_radii):
        tmp_cfg = BlobPseudoCfg(**{**cfg.__dict__,
                                   'structural_smooth_method': 'none' if radius == 0 else 'tophat',
                                   'structural_tophat_radius': radius})
        processed = smooth_structural_channel(demo_struct_raw, tmp_cfg)
        resp = meijering_response(processed, cfg.dendrite_sigmas)
        nz_resp = resp[resp > 0]
        thr = float(threshold_otsu(nz_resp)) if nz_resp.size else 0.0
        dmask = resp > thr

        axes[0, j].imshow(processed, cmap='gray', vmin=0, vmax=0.5)
        axes[0, j].set_title(f'r={radius}' if radius > 0 else 'raw (no filter)', fontsize=9)
        axes[0, j].axis('off')

        axes[1, j].imshow(resp, cmap='hot')
        axes[1, j].set_title(f'Meijering response', fontsize=9)
        axes[1, j].axis('off')

        axes[2, j].imshow(dmask, cmap='gray')
        axes[2, j].set_title(f'mask (Otsu={thr:.3f}, frac={dmask.mean():.2%})', fontsize=9)
        axes[2, j].axis('off')

    axes[0, 0].set_ylabel('structural ch', fontsize=10)
    axes[1, 0].set_ylabel('Meijering', fontsize=10)
    axes[2, 0].set_ylabel('binary mask', fontsize=10)
    fig.suptitle(
        f'White top-hat radius sweep  '
        f'(dendrite_sigmas={list(cfg.dendrite_sigmas)}, per-image Otsu)',
        fontsize=11,
    )
    fig.tight_layout()
    save_fig(fig, '22_sweep_tophat_radius')
    plt.show()
else:
    print('SWEEP is False -- skipping top-hat radius sweep')

## Sweep: Dendrites (density method)

**Guarded by `SWEEP = True`.**

Sweep `density_input` × `density_sigma` to pick a σ that bridges
inter-punctum spacing without merging adjacent neurites. Skeleton ON
vs OFF is shown below.

In [ ]:
if SWEEP:
    _base = {**cfg.__dict__, 'dendrite_method': 'density'}
    _inputs = ['raw', 'tophat', 'puncta']
    _sigmas = [3.0, 6.0, 9.0, 12.0]

    fig, axes = plt.subplots(len(_inputs), len(_sigmas), figsize=(4 * len(_sigmas), 4 * len(_inputs)))
    for i, inp in enumerate(_inputs):
        for j, sig in enumerate(_sigmas):
            tmp = BlobPseudoCfg(**{**_base, 'density_input': inp, 'density_sigma': sig})
            d = make_density_dendrite_mask(demo_struct_channel, tmp)
            ax = axes[i, j] if len(_inputs) > 1 else axes[j]
            show_mask_overlay(
                demo_struct_channel, d['dendrite_mask'],
                ax, color=(1.0, 0.55, 0.0), alpha=0.45, vmax=0.5,
            )
            ax.set_title(
                f"input='{inp}', σ={sig}\n"
                f"thr={d['dendrite_threshold']:.3f}, frac={d['dendrite_mask'].mean():.2%}",
                fontsize=9,
            )
            ax.axis('off')
    fig.suptitle(
        f'Density-method sweep  (image_index={demo_image_index}, '
        f'skeleton={cfg.density_use_skeleton}, prune={cfg.density_skeleton_prune})',
        fontsize=11,
    )
    fig.tight_layout()
    save_fig(fig, '23_sweep_density_input_sigma')
    plt.show()

    # Skeleton ON vs OFF at the recommended default
    fig2, axes2 = plt.subplots(1, 2, figsize=(10, 5))
    for ax, use_skel in zip(axes2, [False, True]):
        tmp = BlobPseudoCfg(**{**_base, 'density_use_skeleton': use_skel})
        d = make_density_dendrite_mask(demo_struct_channel, tmp)
        show_mask_overlay(
            demo_struct_channel, d['dendrite_mask'],
            ax, color=(1.0, 0.55, 0.0), alpha=0.45, vmax=0.5,
        )
        ax.set_title(
            f"skeleton={use_skel}, frac={d['dendrite_mask'].mean():.2%}",
            fontsize=10,
        )
        ax.axis('off')
    fig2.suptitle('Skeleton ON vs OFF (density method)', fontsize=11)
    fig2.tight_layout()
    save_fig(fig2, '24_sweep_density_skeleton')
    plt.show()
else:
    print('SWEEP is False -- skipping density-method sweep')

## Sweep: Blobs (z-score threshold)

**Guarded by `SWEEP = True`.**

Sweep `zscore_threshold` over the canonical sanity batch and a few
additional random patches to see how recall changes. Use the curve
to pick a value that drops most noise without removing obvious
puncta.

In [ ]:
if SWEEP:
    sweep_z = [1.0, 1.5, 2.0, 3.0, 5.0]
    n_sweep_imgs = min(20, len(patch_records))
    sweep_idx = sorted(set(SANITY_INDICES) | set(
        np.random.default_rng(SEED).choice(
            len(patch_records), n_sweep_imgs, replace=False,
        ).tolist()
    ))
    sweep_patches = {i: get_patch(i) for i in sorted(
        sweep_idx, key=lambda j: int(patch_records[j]['image_index'])
    )}

    rows = []
    print(f'{"z":>5s} | {"pre kept":>10s} {"post kept":>10s}')
    for z in sweep_z:
        cfg_t = BlobPseudoCfg(**{**cfg.__dict__, 'zscore_threshold': z})
        n_pre = n_post = 0
        for i in sweep_idx:
            p = sweep_patches[i]
            b_pre  = detect_blobs_log(p[cfg.pre_channel],  cfg_t)
            b_post = detect_blobs_log(p[cfg.post_channel], cfg_t)
            n_pre  += sum(s['kept'] for s in score_blobs_zscore(p[cfg.pre_channel],  b_pre,  cfg_t))
            n_post += sum(s['kept'] for s in score_blobs_zscore(p[cfg.post_channel], b_post, cfg_t))
        rows.append((z, n_pre, n_post))
        print(f'{z:5.1f} | {n_pre:10d} {n_post:10d}')

    fig, ax = plt.subplots(figsize=(8, 4))
    zs    = [r[0] for r in rows]
    preks = [r[1] for r in rows]
    posks = [r[2] for r in rows]
    ax.plot(zs, preks, marker='o', color='C2', label='pre kept')
    ax.plot(zs, posks, marker='s', color='C3', label='post kept')
    ax.axvline(cfg.zscore_threshold, color='k', ls=':', alpha=0.6,
               label=f'cfg.zscore_threshold = {cfg.zscore_threshold}')
    ax.set_xlabel('z-score threshold')
    ax.set_ylabel('# blobs kept (sum across sweep patches)')
    ax.set_title(f'z-score threshold sweep  (n={len(sweep_idx)} patches)')
    ax.legend(); ax.grid(alpha=0.3)
    fig.tight_layout()
    save_fig(fig, '23_sweep_zscore_threshold')
    plt.show()
else:
    print('SWEEP is False -- skipping z-score threshold sweep')

---
## Reconfiguration

After reviewing the sweeps above, override the parameters that
need changing. Then clear the structural-mask cache so downstream
cells recompute with the new values.

In [ ]:
# Uncomment + edit, then run this cell.
# The cache clear forces all downstream cells to recompute.
#
# cfg.soma_intensity_percentile = 90.0
# cfg.dendrite_threshold        = None  # None = per-image Otsu (default); set a float to fix
# cfg.structural_tophat_radius  = 15
# cfg.zscore_threshold          = 2.0
# # Density-method overrides:
# cfg.dendrite_method            = 'density'
# cfg.density_input              = 'tophat'
# cfg.density_sigma              = 6.0
# cfg.density_global_threshold   = None  # None = per-image Otsu
# cfg.density_use_skeleton       = True
# cfg.density_skeleton_prune     = 10
# cfg.density_skeleton_dilate    = 2
# _struct_full_cache.clear()
# cache.clear()
# logger.info(
#     f'overrides applied: dendrite_threshold={cfg.dendrite_threshold}, '
#     f'soma_intensity_percentile={cfg.soma_intensity_percentile}, '
#     f'structural_tophat_radius={cfg.structural_tophat_radius}, '
#     f'zscore_threshold={cfg.zscore_threshold}; '
#     'caches cleared'
# )
print('Current cfg:')
print(cfg)

### Test on 4 full images

Run the full pipeline on 4 randomly-chosen source images and show
the reassembled pseudo-label overlay. This is the final validation
before committing to the full dataset generation.

In [ ]:
rng_test = np.random.default_rng(SEED + 99)
test_image_indices = rng_test.choice(
    available_image_indices,
    size=min(4, len(available_image_indices)),
    replace=False,
).tolist()
logger.info(f'test images: {test_image_indices}')

for img_idx in test_image_indices:
    full_image = get_full_image(int(img_idx))
    _struct_full_cache.pop(int(img_idx), None)  # force recompute with current cfg
    full_struct = get_full_struct(int(img_idx))

    # Run pipeline on every patch of this image.
    C, H_full, W_full = full_image.shape
    ps = int(patch_records[image_to_patch_positions[img_idx][0]]['patch_size'])
    label_full = np.zeros((H_full, W_full), dtype=np.float32)
    n_blobs_total = 0

    for pos in image_to_patch_positions[img_idx]:
        rec = patch_records[pos]
        y0 = int(rec['grid_row']) * ps
        x0 = int(rec['grid_col']) * ps
        patch = full_image[:, y0:y0 + ps, x0:x0 + ps]
        sl = (slice(y0, y0 + ps), slice(x0, x0 + ps))
        # Dynamic slice: pick every 2D full-shape array.
        _full_shape = full_struct['full_image'].shape[-2:]
        struct_slice = {
            k: v[sl] for k, v in full_struct.items()
            if hasattr(v, 'shape') and v.ndim == 2 and v.shape[-2:] == _full_shape
        }
        lbl, _, st = generate_blob_pseudolabel(
            patch, cfg, precomputed_struct=struct_slice,
        )
        label_full[y0:y0 + ps, x0:x0 + ps] = lbl
        n_blobs_total += st['px_label']

    # Show full-image overlay.
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    composite = full_image.max(0)
    axes[0].imshow(composite, cmap='gray')
    axes[0].set_title('max-projection composite')
    show_mask_overlay(
        composite, full_struct['near_structural'],
        axes[1], color=(0.2, 0.7, 1.0), alpha=0.3,
    )
    axes[1].set_title('structural mask overlay')
    show_mask_overlay(
        composite, label_full > 0,
        axes[2], color=(1, 0.2, 0.2), alpha=0.6,
    )
    axes[2].set_title(
        f'pseudo-label overlay  '
        f'(label_px={n_blobs_total}, frac={label_full.mean():.3%})'
    )
    for ax in axes.flat:
        ax.axis('off')
    fig.suptitle(
        f'Full-image test  image_index={img_idx}  '
        f'({len(image_to_patch_positions[img_idx])} patches)',
        fontsize=11,
    )
    fig.tight_layout()
    save_fig(fig, f'30_test_fullimage_{img_idx}')
    plt.show()

---
## Full labels pipeline

**Guarded by `GENERATE_ALL = True`.**

Each pseudo-label is saved as `(128, 128)` `uint8` (0/1) to
`OUTPUT_ROOT/<original_filename>.npy`. This matches the
`PseudoLabelSegDataset` mask convention (`(H, W)`; the dataset adds
the channel axis). To reuse these files in a training notebook,
eager-load them into a dict and pass as `precomputed_masks`:

```python
precomputed = {rec['filename']: np.load(OUTPUT_ROOT / rec['filename'])
               for rec in patch_ds.records}
PseudoLabelSegDataset(..., precomputed_masks=precomputed)
```

Per-patch stats are collected for the after-run analysis below.

In [ ]:
if GENERATE_ALL:
    all_stats = []
    full_labels_by_image: dict[int, dict[str, np.ndarray]] = {}
    with tqdm(total=len(patch_records), desc='generating pseudo-labels') as pbar:
        for img_idx in available_image_indices:
            full_image, _ = reassemble_image(PATCH_ROOT, int(img_idx))
            img_records = [patch_records[pos] for pos in image_to_patch_positions[img_idx]]

            # full-image LoG + z-score + render + shape + structural gate;
            # avoids per-patch border dead zones at every tile seam.
            labels, stats_by_name = generate_pseudolabels_fullimage(
                full_image, img_records, cfg,
            )
            full_labels_by_image[int(img_idx)] = labels

            for rec in img_records:
                name = rec['filename']
                np.save(OUTPUT_ROOT / name, labels[name].astype(np.uint8))
                st = dict(stats_by_name[name])
                st['filename'] = name
                st['image_index'] = int(img_idx)
                all_stats.append(st)
                pbar.update(1)

            del full_image

    (save_dir / 'per_patch_stats.json').write_text(
        json.dumps([{k: (float(v) if isinstance(v, (np.floating,)) else v)
                      for k, v in s.items()} for s in all_stats], indent=2, default=str)
    )
    logger.info(f'saved {len(all_stats)} pseudo-label files to {OUTPUT_ROOT}')
    logger.info(f'per_patch_stats.json written')
else:
    print('GENERATE_ALL is False -- skipping full pipeline')

### After-run analysis

Bigger visualisations after the full run, computed across many
patches. These are the figures intended for the thesis.

In [ ]:
if GENERATE_ALL:
    # ── Visual review on N random patches ──
    # Loads the saved (H, W) uint8 labels back from disk so the visual
    # review shows the *production* labels, not a per-patch regeneration
    # (which would reintroduce per-tile LoG border bias).
    rng = np.random.default_rng(SEED + 1)
    review_idx = rng.choice(len(patch_records), size=min(N_VISUAL_REVIEW, len(patch_records)), replace=False)
    review_idx = sorted(review_idx, key=lambda j: int(patch_records[int(j)]['image_index']))

    ncols = 4
    nrows = int(np.ceil(len(review_idx) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.2 * ncols, 4.2 * nrows))
    axes = np.atleast_1d(axes).ravel()
    stats_by_name = {s['filename']: s for s in all_stats}
    for ax, i in zip(axes, review_idx):
        rec = patch_records[int(i)]
        name = rec['filename']
        p = get_patch(int(i))
        lbl = np.load(OUTPUT_ROOT / name)
        st = stats_by_name[name]
        show_mask_overlay(p.max(0), lbl, ax, color=(1, 0.2, 0.2), alpha=0.6)
        short = name if len(name) < 36 else name[:16] + '...' + name[-16:]
        ax.set_title(
            f'idx={i}  {short}\n'
            f'label_px={st["px_label"]}  frac={st["frac_label"]:.3%}',
            fontsize=8,
        )
    for ax in axes[len(review_idx):]:
        ax.axis('off')
    fig.suptitle(
        f'Visual review  (n={len(review_idx)} random patches)', fontsize=12,
    )
    fig.tight_layout()
    save_fig(fig, '40_visual_review_grid')
    plt.show()

In [ ]:
if GENERATE_ALL:
    # ── Aggregate statistics ──
    label_pxs   = np.array([s['px_label']             for s in all_stats])
    label_fracs = np.array([s['frac_label']           for s in all_stats])
    n_pre_kept  = np.array([s['n_pre_kept']           for s in all_stats])
    n_post_kept = np.array([s['n_post_kept']          for s in all_stats])
    near_fracs  = np.array([s['frac_near_structural'] for s in all_stats])

    summary = {
        'n_patches': len(all_stats),
        'patches_with_label': int((label_pxs > 0).sum()),
        'patches_with_label_frac': float((label_pxs > 0).mean()),
        'label_frac_median': float(np.median(label_fracs)),
        'label_frac_mean': float(label_fracs.mean()),
        'label_frac_max': float(label_fracs.max()),
        'pre_kept_median':  int(np.median(n_pre_kept)),
        'pre_kept_max':     int(n_pre_kept.max()),
        'post_kept_median': int(np.median(n_post_kept)),
        'post_kept_max':    int(n_post_kept.max()),
        'structural_area_median': float(np.median(near_fracs)),
        'structural_area_mean':   float(near_fracs.mean()),
    }
    (save_dir / 'aggregate_summary.json').write_text(json.dumps(summary, indent=2))
    for k, v in summary.items():
        print(f'{k:30s}: {v}')
    logger.info('aggregate_summary.json written')

In [ ]:
if GENERATE_ALL:
    fig, axes = plt.subplots(2, 2, figsize=(13, 8))
    axes[0, 0].hist(label_fracs * 100, bins=40, edgecolor='k', alpha=0.7, color='C3')
    axes[0, 0].set_xlabel('% pixels labelled')
    axes[0, 0].set_title(
        f'Label sparsity per patch  '
        f'(median={np.median(label_fracs):.3%})'
    )
    axes[0, 1].hist(near_fracs * 100, bins=40, edgecolor='k', alpha=0.7, color='C0')
    axes[0, 1].set_xlabel('% pixels in structural zone')
    axes[0, 1].set_title(
        f'Structural-mask coverage  '
        f'(median={np.median(near_fracs):.2%})'
    )
    axes[1, 0].hist(n_pre_kept, bins=30, edgecolor='k', alpha=0.7, color='C2')
    axes[1, 0].set_xlabel('# pre puncta')
    axes[1, 0].set_title(f'Kept pre per patch  (median={int(np.median(n_pre_kept))})')
    axes[1, 1].hist(n_post_kept, bins=30, edgecolor='k', alpha=0.7, color='C3')
    axes[1, 1].set_xlabel('# post puncta')
    axes[1, 1].set_title(f'Kept post per patch  (median={int(np.median(n_post_kept))})')
    for ax in axes.flat:
        ax.grid(alpha=0.3)
    fig.suptitle(
        f'Aggregate pseudo-label statistics  (n={len(all_stats)} patches)',
        fontsize=12,
    )
    fig.tight_layout()
    save_fig(fig, '41_aggregate_statistics')
    plt.show()

In [ ]:
if GENERATE_ALL:
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.scatter(near_fracs * 100, label_fracs * 100, s=8, alpha=0.4, color='C3')
    ax.set_xlabel('% pixels in structural zone')
    ax.set_ylabel('% pixels labelled')
    ax.set_title(
        f'Label sparsity vs structural coverage  '
        f'(n={len(all_stats)})'
    )
    ax.grid(alpha=0.3)
    fig.tight_layout()
    save_fig(fig, '42_sparsity_vs_coverage')
    plt.show()

## References

* Wang, Y. et al. *SynQuant: an automatic tool to quantify synapses
  from fluorescence microscopy images.* Bioinformatics 36(5):1599 (2020).
* Meijering, E. et al. *Design and validation of a tool for neurite
  tracing and analysis in fluorescence microscopy images.*
  Cytometry A 58:167 (2004).
* Lindeberg, T. *Feature detection with automatic scale selection.*
  IJCV 30:79 (1998).
* Frangi, A. F. et al. *Multiscale vessel enhancement filtering.*
  MICCAI 1998.
* Xiao, R. et al. *DDeep3M+: adaptive enhancement powered weakly
  supervised learning for neuron segmentation.* Neurophotonics 10(3) (2023).
* Fantuzzo, J. A. et al. *Intellicount: high-throughput quantification
  of fluorescent synaptic protein puncta by machine learning.* eNeuro (2017).
* Huang, Q. et al. *Weakly Supervised Learning of 3D Deep Network for
  Neuron Reconstruction.* Front. Neuroanat. 14:38 (2020).
* Pathak, S. et al. *Nanotopographic control of actin waves and
  growth cone navigation in developing neurons.*
  Front. Cell Dev. Biol. 13:1631520 (2025).